# Bank Complaints: 03 AI Labels (Q5)

"Managing an account" is the biggest complaint issue (3,836 complaints) and too vague to act on. I hand-labelled 200 of them, then tested whether a small local AI model could do the same job before trusting it.

- **Models:** Llama 3.2 3B and Qwen 2.5 3B, run locally with Ollama (free, nothing leaves the laptop)
- **Answer key:** my labels in `data/processed/label_sample_200.xlsx`
- **Setup:** install Ollama, `ollama pull qwen2.5:3b`, `pip install ollama openpyxl`

In [1]:
import pandas as pd
import ollama
import time

## 1. Labels and prompts

I kept every prompt version so I could compare them: **v1** is plain definitions, **v2** adds sharper rules and examples, **v3** asks the model to answer questions in order.

In [2]:
labels = ["fraud_scam", "frozen_closed", "funds_held", "fees", "access_login", "other"]

prompts = {}

prompts["v1"] = """You label bank customer complaints. Reply with exactly one label from this list and nothing else:
fraud_scam: money taken without permission, scam, hacked account
frozen_closed: the bank froze, restricted, or closed the account
funds_held: money stuck, on hold, missing, or a transfer not received
fees: overdraft or other fees charged
access_login: cannot log in, locked out, ID verification problems, card blocked
other: none of the above
If more than one fits, choose the problem the customer most wants fixed."""

prompts["v2"] = """You label bank customer complaints. Reply with ONLY one label, no other words.

Labels and how to choose:
fraud_scam = someone took money without permission, a scam, or the account was hacked.
frozen_closed = the BANK froze, restricted, suspended, or closed the account.
funds_held = a specific deposit, check, or transfer is on hold, delayed, or missing, but the account itself is still open.
fees = the main complaint is a fee, overdraft charge, or penalty.
access_login = the customer cannot log in, is locked out, ID or identity verification fails, or the card is blocked or declined.
other = customer service, errors, or anything that does not clearly fit above. Use other when unsure.

Examples:
"Someone made Zelle transfers I never authorized" -> fraud_scam
"The bank closed my account without explanation and kept my money" -> frozen_closed
"My mobile check deposit has been on hold for two weeks" -> funds_held
"I was charged three $35 overdraft fees in one day" -> fees
"I cannot log into the app and they keep rejecting my ID" -> access_login
"The branch manager was rude and gave me wrong information" -> other"""

prompts["v3"] = """You label bank customer complaints. Answer with ONLY the label, no other words.

Go through these questions IN ORDER and stop at the first YES:
1. Did someone else (not the bank) take the customer's money without permission, scam them, or did the bank deny their dispute about unauthorized transactions? -> fraud_scam
2. Did the BANK freeze, restrict, lock, suspend, put on hold, or close the customer's whole account? -> frozen_closed
3. Is a specific deposit, check, transfer, or refund on hold, delayed, reversed, or missing? -> funds_held
4. Is the main complaint a fee or overdraft charge? -> fees
5. Can the customer not log in, or is their card blocked or declined, while the account is still open? -> access_login
6. Otherwise -> other

A refund dispute with a merchant or seller is NOT fraud_scam. It is other.
A mistake made by the bank itself is NOT fraud_scam. It is other."""

## 2. Classifier

One function for every test. It sends the rules and one complaint to the model and returns one label.
- only the first 800 characters, and a reply of 10 tokens at most, to keep it fast on an 8 GB laptop
- `temperature: 0` so the same complaint gets the same answer every time
- `"invalid"` if the reply has none of the labels

In [3]:
def classify(text, prompt, model="llama3.2:3b"):
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": text[:800]},
        ],
        options={"temperature": 0, "num_predict": 10},
    )
    answer = response["message"]["content"].strip().lower()
    for label in labels:
        if label in answer:
            return label
    return "invalid"

## 3. Load the labels

- `my_label`: my first pass
- `reviewed_label`: after a second check against the rules (an AI assistant flagged rule breaks, I checked each one; 59 of 200 changed, reason in `review_note`). All tests use this column.
- **dev** (rows 0-99) is for comparing prompts. **test** (rows 100-199) is used once at the end, so the final number isn't tuned to it.

In [4]:
df = pd.read_excel(r"..\data\processed\label_sample_200.xlsx")
dev = df.iloc[:100].copy()
test = df.iloc[100:200].copy()
print(len(dev), "dev rows,", len(test), "test rows")
df["reviewed_label"].value_counts()

100 dev rows, 100 test rows


reviewed_label
fraud_scam       61
other            42
frozen_closed    41
funds_held       34
access_login     21
fees              1
Name: count, dtype: int64

## 4. Evaluate

Accuracy is the share of complaints where the model matches my label. The crosstab shows where it goes wrong: my label down the side, the model's across the top.

In [5]:
def evaluate(data, version, model="llama3.2:3b"):
    col = f"ai_{version}_{model.split(':')[0]}"
    start = time.time()
    data[col] = data["narrative"].apply(lambda t: classify(t, prompts[version], model))
    seconds = round(time.time() - start)
    accuracy = round((data["reviewed_label"] == data[col]).mean() * 100, 1)
    print(f"{version} ({model}): {accuracy}% accuracy on {len(data)} rows, {seconds} seconds")
    return pd.crosstab(data["reviewed_label"], data[col], margins=True, margins_name="Total")

### Dev: v2 vs v3 on rows 0-99

In [6]:
evaluate(dev, "v2", model="qwen2.5:3b")

v2 (qwen2.5:3b): 48.0% accuracy on 100 rows, 797 seconds


ai_v2_qwen2.5,access_login,fees,fraud_scam,frozen_closed,funds_held,other,Total
reviewed_label,,,,,,,
access_login,4,3,1,0,0,1,9
fees,0,1,0,0,0,0,1
fraud_scam,0,1,21,2,1,2,27
frozen_closed,1,2,3,9,0,3,18
funds_held,1,9,2,3,0,2,17
other,0,8,4,3,0,13,28
Total,6,24,31,17,1,21,100


In [7]:
evaluate(dev, "v3", model="qwen2.5:3b")

v3 (qwen2.5:3b): 54.0% accuracy on 100 rows, 935 seconds


ai_v3_qwen2.5,access_login,fees,fraud_scam,frozen_closed,funds_held,invalid,other,Total
reviewed_label,,,,,,,,
access_login,3,0,1,2,0,1,2,9
fees,0,0,0,0,0,0,1,1
fraud_scam,0,0,22,1,2,0,2,27
frozen_closed,1,0,5,7,1,1,3,18
funds_held,0,1,0,1,4,0,11,17
other,0,3,5,2,0,0,18,28
Total,4,4,33,13,7,2,37,100


### Test: the dev winner, once, on rows 100-199
v3 beat v2 on dev (54% vs 48%), so v3 goes to the test.

In [8]:
evaluate(test, "v3", model="qwen2.5:3b")

v3 (qwen2.5:3b): 71.0% accuracy on 100 rows, 824 seconds


ai_v3_qwen2.5,access_login,fees,fraud_scam,frozen_closed,funds_held,invalid,other,Total
reviewed_label,,,,,,,,
access_login,9,0,0,1,1,0,1,12
fraud_scam,0,0,32,0,1,0,1,34
frozen_closed,0,0,2,16,1,0,4,23
funds_held,0,1,4,2,6,2,2,17
other,0,1,3,1,0,1,8,14
Total,9,2,41,20,9,3,16,100


In [9]:
dev.to_csv(r"..\data\processed\ai_results_dev.csv", index=False)
test.to_csv(r"..\data\processed\ai_results_test.csv", index=False)

### The same tables with totals (no re-run needed)

In [10]:
dev_saved = pd.read_csv(r"..\data\processed\ai_results_dev.csv")
test_saved = pd.read_csv(r"..\data\processed\ai_results_test.csv")
display(pd.crosstab(dev_saved["reviewed_label"], dev_saved["ai_v2_qwen2.5"], margins=True, margins_name="Total"))
display(pd.crosstab(dev_saved["reviewed_label"], dev_saved["ai_v3_qwen2.5"], margins=True, margins_name="Total"))
display(pd.crosstab(test_saved["reviewed_label"], test_saved["ai_v3_qwen2.5"], margins=True, margins_name="Total"))

ai_v2_qwen2.5,access_login,fees,fraud_scam,frozen_closed,funds_held,other,Total
reviewed_label,,,,,,,
access_login,4,3,1,0,0,1,9
fees,0,1,0,0,0,0,1
fraud_scam,0,1,21,2,1,2,27
frozen_closed,1,2,3,9,0,3,18
funds_held,1,9,2,3,0,2,17
other,0,8,4,3,0,13,28
Total,6,24,31,17,1,21,100


ai_v3_qwen2.5,access_login,fees,fraud_scam,frozen_closed,funds_held,invalid,other,Total
reviewed_label,,,,,,,,
access_login,3,0,1,2,0,1,2,9
fees,0,0,0,0,0,0,1,1
fraud_scam,0,0,22,1,2,0,2,27
frozen_closed,1,0,5,7,1,1,3,18
funds_held,0,1,0,1,4,0,11,17
other,0,3,5,2,0,0,18,28
Total,4,4,33,13,7,2,37,100


ai_v3_qwen2.5,access_login,fees,fraud_scam,frozen_closed,funds_held,invalid,other,Total
reviewed_label,,,,,,,,
access_login,9,0,0,1,1,0,1,12
fraud_scam,0,0,32,0,1,0,1,34
frozen_closed,0,0,2,16,1,0,4,23
funds_held,0,1,4,2,6,2,2,17
other,0,1,3,1,0,1,8,14
Total,9,2,41,20,9,3,16,100


## 5. Results

| Labels | Prompt | Model | Rows | Accuracy |
|---|---|---|---|---|
| first pass | v1 | llama3.2:3b | 0-99 | 31% |
| first pass | v2 | llama3.2:3b | 0-99 | 35% |
| first pass | v2 | qwen2.5:3b | 0-99 | 41% |
| reviewed | v2 | qwen2.5:3b | dev 0-99 | 48% |
| reviewed | v3 | qwen2.5:3b | dev 0-99 | 54% |
| reviewed | v3 | qwen2.5:3b | **test 100-199** | **71%** |

I set the bar at 70% before testing. 71% clears it, but only just: with 100 rows the real number could be anywhere from about 62% to 80%, and dev was 54%. So I use the model as a first sorter that a person checks, and the Q5 findings come from my reviewed labels.

The biggest gains came from fixing my own labels and giving the model questions in order, not from switching models.